In [0]:
%pip install -U -qqqq databricks-agents>=0.16.0 mlflow>=2.20.2

In [0]:
%pip install -U -qqqq databricks-langchain

In [0]:
%pip install -U -qqqq databricks-openai

In [0]:
%pip install -U -qqqq databricks-ai-bridge

Use ChatAgent to author agents
Databricks recommends using MLflow's ChatAgent interface to author production-grade agents. This chat schema specification is designed for agent scenarios and is similar to, but not strictly compatible with, the OpenAI ChatCompletion schema. ChatAgent also adds functionality for multi-turn, tool-calling agents.

Authoring your agent using ChatAgent provides the following benefits:

Advanced agent capabilities

Streaming output: Enable interactive user experiences by streaming output in smaller chunks.

Comprehensive tool-calling message history: Return multiple messages, including intermediate tool-calling messages, for improved quality and conversation management.

Tool-calling confirmation support

Multi-agent system support

Streamlined development, deployment, and monitoring

Framework agnostic Databricks feature integration: Write your agent in any framework of your choice and get out-of-the-box compatibility with AI Playground, Agent Evaluation, and Agent Monitoring.

Typed authoring interfaces: Write agent code using typed Python classes, benefiting from IDE and notebook autocomplete.

Automatic signature inference: MLflow automatically infers ChatAgent signatures when logging the agent, simplifying registration and deployment. See Infer Model Signature during logging.

AI Gateway-enhanced inference tables: AI Gateway inference tables are automatically enabled for deployed agents, providing access to detailed request log metadata.

In [0]:
%pip install -U -qqqq mlflow databricks-langchain databricks-agents uv langgraph==0.3.4
dbutils.library.restartPython()

In [0]:
%pip install mlflow --upgrade --pre

In [0]:
%pip install mlflow databricks-agents --upgrade --pre

####LangGraph tool-calling agent

In [0]:
%pip install -U -qqqq mlflow databricks-langchain databricks-agents uv langgraph==0.3.4
dbutils.library.restartPython()

In [0]:
from typing import Any, Generator, Optional, Sequence, Union

import mlflow
from databricks_langchain import ChatDatabricks
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = ""

############################################
# Define local tools (no UC)
############################################
@tool
def multiply_by_two(number: int) -> str:
    """Multiply an integer by two."""
    return f"{number} times two is {number * 2}"

@tool
def greet_user(name: str) -> str:
    """Return a friendly greeting to the user."""
    return f"Hello, {name}! Nice to meet you."

tools: list[BaseTool] = [multiply_by_two, greet_user]

############################################
# Define agent logic using LangGraph
############################################
def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    def should_continue(state: ChatAgentState):
        messages = state["messages"]
        last_message = messages[-1]
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    preprocessor = (
        RunnableLambda(lambda state: [{"role": "system", "content": system_prompt}] + state["messages"])
        if system_prompt else RunnableLambda(lambda state: state["messages"])
    )
    model_runnable = preprocessor | model

    def call_model(state: ChatAgentState, config: RunnableConfig):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)
    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges("agent", should_continue, {
        "continue": "tools",
        "end": END,
    })
    workflow.add_edge("tools", "agent")

    return workflow.compile()

############################################
# Agent wrapper class
############################################
class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}
        messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                messages.extend(ChatAgentMessage(**msg) for msg in node_data.get("messages", []))
        return ChatAgentResponse(messages=messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> Generator[ChatAgentChunk, None, None]:
        request = {"messages": self._convert_messages_to_dict(messages)}
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (
                    ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"]
                )

############################################
# Create and register the agent
############################################
mlflow.langchain.autolog()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphChatAgent(agent)
mlflow.models.set_model(AGENT)


In [0]:
from mlflow.types.agent import ChatAgentMessage

messages = [ChatAgentMessage(role="user", content="What is 6 multiplied by two?")]
response = AGENT.predict(messages=messages)
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")

messages = [ChatAgentMessage(role="user", content="Say hello to Alice")]
response = AGENT.predict(messages=messages)
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")

####OpenAI tool-calling agent

In [0]:

from agent import AGENT

def test_multiply():
    print("Test: multiply_by_two")
    question = "What is 9 multiplied by two?"
    response = AGENT.run(question)
    print("Response:", response)

def test_greet():
    print("\nTest: greet_user")
    question = "Say hello to Jordan"
    response = AGENT.run(question)
    print("Response:", response)

if __name__ == "__main__":
    test_multiply()
    test_greet()



In [0]:
%pip install -U -qqqq mlflow backoff databricks-openai databricks-agents uv
dbutils.library.restartPython()

In [0]:
import json
import backoff
import mlflow
from uuid import uuid4
from typing import List, Optional, Generator, Any, Callable, Dict
from databricks.sdk import WorkspaceClient
from openai import OpenAI
from mlflow.entities import SpanType
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentMessage,
    ChatAgentChunk,
    ChatAgentResponse,
    ChatContext,
)
from pydantic import BaseModel

# ✅ LLM Endpoint
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
SYSTEM_PROMPT = "You are a helpful assistant."

# ✅ Define tool schema wrapper
class ToolInfo(BaseModel):
    name: str
    spec: dict
    exec_fn: Callable

# ✅ Define tools
def multiply_by_two(number: int) -> str:
    return f"{number} times two is {number * 2}"

def greet_user(name: str) -> str:
    return f"Hello, {name}! Nice to meet you."

TOOLS: List[ToolInfo] = [
    ToolInfo(
        name="multiply_by_two",
        spec={
            "type": "function",
            "function": {
                "name": "multiply_by_two",
                "description": "Multiply a number by two.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "number": {
                            "type": "integer",
                            "description": "Number to double"
                        }
                    },
                    "required": ["number"]
                }
            }
        },
        exec_fn=multiply_by_two,
    ),
    ToolInfo(
        name="greet_user",
        spec={
            "type": "function",
            "function": {
                "name": "greet_user",
                "description": "Greet a user by name.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "User's name"
                        }
                    },
                    "required": ["name"]
                }
            }
        },
        exec_fn=greet_user,
    ),
]

# ✅ Define the Agent
class ToolCallingAgent(ChatAgent):
    def __init__(self, llm_endpoint: str, tools: List[ToolInfo]):
        super().__init__()
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client: OpenAI = self.workspace_client.serving_endpoints.get_open_ai_client()
        self._tools_dict = {tool.name: tool for tool in tools}

    def get_tool_specs(self):
        return [tool.spec for tool in self._tools_dict.values()]

    def execute_tool(self, tool_name: str, args: dict) -> Any:
        if tool_name not in self._tools_dict:
            raise ValueError(f"Unknown tool: {tool_name}")
        return self._tools_dict[tool_name].exec_fn(**args)

    def prepare_messages_for_llm(self, messages: List[ChatAgentMessage]) -> List[Dict[str, Any]]:
        result = []
        for msg in messages:
            msg_dict = msg.model_dump_compat(exclude_none=True)
            msg_dict.pop("id", None)  # Remove id
            if msg_dict.get("role") != "tool":
                msg_dict.pop("name", None)  # Keep 'name' only for role='tool'
            result.append(msg_dict)
        return result

    @mlflow.trace(span_type=SpanType.AGENT)
    def predict(self, messages: List[ChatAgentMessage], context: Optional[ChatContext] = None, custom_inputs: Optional[dict[str, Any]] = None) -> ChatAgentResponse:
        return ChatAgentResponse(messages=[chunk.delta for chunk in self.predict_stream(messages)])

    @mlflow.trace(span_type=SpanType.AGENT)
    def predict_stream(self, messages: List[ChatAgentMessage], context: Optional[ChatContext] = None, custom_inputs: Optional[dict[str, Any]] = None) -> Generator[ChatAgentChunk, None, None]:
        history = [ChatAgentMessage(role="system", content=SYSTEM_PROMPT)] + messages
        for msg in self.call_and_run_tools(history):
            yield ChatAgentChunk(delta=msg)

    @backoff.on_exception(backoff.expo, Exception)
    def chat_completion(self, messages: List[ChatAgentMessage]):
        return self.model_serving_client.chat.completions.create(
            model=self.llm_endpoint,
            messages=self.prepare_messages_for_llm(messages),
            tools=self.get_tool_specs()
        )

    def call_and_run_tools(self, messages: List[ChatAgentMessage], max_iter: int = 5) -> Generator[ChatAgentMessage, None, None]:
        history = messages.copy()
        for _ in range(max_iter):
            response = self.chat_completion(history)
            raw_msg = response.choices[0].message
            assistant_msg = ChatAgentMessage(**raw_msg.to_dict(), id=str(uuid4()))  # ✅ Add ID
            history.append(assistant_msg)
            yield assistant_msg

            if not assistant_msg.tool_calls:
                return

            for call in assistant_msg.tool_calls:
                args = json.loads(call.function.arguments)
                result = str(self.execute_tool(call.function.name, args))
                tool_msg = ChatAgentMessage(
                    role="tool",
                    name=call.function.name,
                    tool_call_id=call.id,
                    content=result,
                    id=str(uuid4())  # ✅ Add ID
                )
                history.append(tool_msg)
                yield tool_msg

        yield ChatAgentMessage(
            content="I'm sorry, I couldn't determine the answer after several attempts.",
            role="assistant",
            id=str(uuid4())  # ✅ Add ID
        )

# ✅ Register the agent
mlflow.openai.autolog()
AGENT = ToolCallingAgent(llm_endpoint=LLM_ENDPOINT_NAME, tools=TOOLS)
mlflow.models.set_model(AGENT)


In [0]:
from mlflow.types.agent import ChatAgentMessage

# 🔢 Test: multiply_by_two
print("🔢 Test: multiply_by_two")
messages = [ChatAgentMessage(role="user", content="What is 8 multiplied by two?")]
response = AGENT.predict(messages=messages)
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")

# 👋 Test: greet_user
print("\n👋 Test: greet_user")
messages = [ChatAgentMessage(role="user", content="Say hello to Ava")]
response = AGENT.predict(messages=messages)
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")



####AutoGen tool-calling agent

Mosaic AI Agent Framework: Author and deploy a tool-calling AutoGen agent
This notebook demonstrates how to author an AutoGen agent that's compatible with Mosaic AI Agent Framework features. In this notebook, you learn to:

Author a tool-calling AutoGen agent wrapped with ChatAgent with custom inputs and outputs
Manually test the agent's output
Evaluate the agent using Mosaic AI Agent Evaluation
Log and deploy the agent
To learn more about authoring an agent using Mosaic AI Agent Framework, see Databricks documentation (AWS | Azure).

Prerequisites
A cluster that has access to both Unity Catalog and the ML Runtime.
Permissions are required to create models within a catalog and schema in Unity Catalog.
Permission to create a serving endpoint.

Wrap the AutoGen agent using the ChatAgent interface
For compatibility with Databricks AI features, the AutogenAgent class implements the ChatAgent interface to wrap the AutoGen agent. The predict_stream method is not implemented in ChatAgent because streaming is not supported in Autogen version 0.2 for custom models.

Databricks recommends using ChatAgent as it simplifies authoring multi-turn conversational agents using an open source standard. See MLflow's ChatAgent documentation.

In [0]:
!pip install autogen-agentchat==0.2.40 unitycatalog-autogen[databricks] databricks-ai-bridge uv databricks-agents
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
from typing import List, Optional

# ✅ DatabricksModelServingClient: A simple wrapper around Databricks Model Serving endpoints
# Inspired by https://microsoft.github.io/autogen/0.2/blog/2024/01/26/Custom-Models

class DatabricksModelServingClient:
    def __init__(self, config, **kwargs):
        self.workspace = WorkspaceClient()
        self.openai_client = self.workspace.serving_endpoints.get_open_ai_client()
        self.endpoint_name = config.get("endpoint_name")
        self.llm_config = config.get("llm_config", {})
        self.config = config

    def transform_messages(self, input_data, remove_keys=['id']):
        """
        Remove unsupported fields like 'id' from messages.
        """
        output_messages = input_data['messages'].copy()
        for message in output_messages:
            for key in remove_keys:
                if key in message:
                    message.pop(key)
        return output_messages

    def create(self, input_data):
        """
        Perform a chat completion request to the Databricks endpoint.
        """
        input_messages = self.transform_messages(input_data)

        response = self.openai_client.chat.completions.create(
            model=self.endpoint_name,
            messages=input_messages,
            tools=input_data.get("tools", []),
            **self.llm_config
        )
        return response

    def message_retrieval(self, response):
        """
        Extract assistant messages from the response.
        """
        return [choice.message for choice in response.choices]

    def cost(self, response):
        """
        Stub for cost tracking (optional).
        """
        return 0

    def get_usage(self, response):
        """
        Return token usage stats.
        """
        usage = response.usage
        return {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens
        }


In [0]:
%pip install --upgrade pyautogen


In [0]:
import json
import os
import uuid
from random import randint
from typing import Optional, Any

import mlflow
from mlflow.types.agent import ChatAgentMessage, ChatContext, ChatAgentResponse
from mlflow.pyfunc import ChatAgent

from databricks.sdk import WorkspaceClient

# ✅ Define the DatabricksModelServingClient inline
class DatabricksModelServingClient:
    def __init__(self, config, **kwargs):
        self.workspace = WorkspaceClient()
        self.openai_client = self.workspace.serving_endpoints.get_open_ai_client()
        self.endpoint_name = config.get("endpoint_name")
        self.llm_config = config.get("llm_config", {})

    def transform_messages(self, input_data, remove_keys=['id']):
        msgs = input_data['messages'].copy()
        for m in msgs:
            for k in remove_keys:
                m.pop(k, None)
        return msgs

    def create(self, input_data):
        messages = self.transform_messages(input_data)
        return self.openai_client.chat.completions.create(
            model=self.endpoint_name,
            messages=messages,
            tools=input_data.get("tools", []),
            **self.llm_config
        )

    def message_retrieval(self, response):
        return [c.message for c in response.choices]

    def get_usage(self, response):
        u = response.usage
        return {
            "prompt_tokens": u.prompt_tokens,
            "completion_tokens": u.completion_tokens,
            "total_tokens": u.total_tokens
        }

# ✅ Required autogen imports
from autogen import ConversableAgent, register_function

# ✅ Enable MLflow tracking
mlflow.autogen.autolog(log_traces=True)

# ✅ Define your Databricks endpoint
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

# ✅ Define sample tools
def generate_random_ints(min: int, max: int, size: int) -> dict[str, Any]:
    """Generate size random ints in the range [min, max]."""
    attachments = {"min": str(min), "max": str(max)}
    custom_outputs = [randint(min, max) for _ in range(size)]
    content = f"Successfully generated array of {size} random ints in [{min}, {max}]."
    return {
        "content": content,
        "attachments": attachments,
        "custom_outputs": {"random_nums": custom_outputs},
    }

def weather_in_california_city(city: str) -> str:
    """Get the weather description of a city in California."""
    return f"The weather in {city} is sunny."

tools = [generate_random_ints, weather_in_california_city]

# ✅ Autogen Agent
class AutogenAgent(ChatAgent):
    def __init__(self, tools=None, model_name="autogen_agent"):
        self.model_name = model_name
        self.tools = tools or []

    def _create_agents(self, chat_history):
        def _is_termination_message(message):
            content = message.get("content", "")
            return ("TERMINATE" in content.upper()) or (message['role'] == 'user' and 'tool_calls' not in message)

        user_proxy = ConversableAgent(
            name="User",
            llm_config=False,
            is_termination_msg=_is_termination_message,
            human_input_mode="NEVER",
        )

        config_list = [{
            "model_client_cls": DatabricksModelServingClient,  # ✅ use class directly
            "model": self.model_name,
            "endpoint_name": LLM_ENDPOINT_NAME,
            "llm_config": {"max_tokens": 1500, "temperature": 0.01}
        }]

        assistant = ConversableAgent(
            name="Assistant",
            system_message="You are a helpful assistant with tools. Use tools first. Only generate if needed.",
            llm_config={"config_list": config_list, "cache_seed": None, "stream": True},
            chat_messages={user_proxy: chat_history}
        )

        for tool in self.tools:
            register_function(
                tool,
                caller=assistant,
                executor=user_proxy,
                description=tool.__doc__,
            )

        return assistant, user_proxy

    def _convert_tool_message(self, message):
        tool_response = message['tool_responses'][0]
        tool_message = {
            'role': 'tool',
            'name': message['name'],
            'tool_call_id': tool_response['tool_call_id'],
        }
        custom_outputs = None
        try:
            tool_response_content = json.loads(tool_response["content"])
            if "custom_outputs" in tool_response_content:
                custom_outputs = tool_response_content["custom_outputs"]
            tool_message['attachments'] = tool_response_content.get('attachments')
            content = tool_response_content.get('content', '')
        except Exception:
            content = tool_response['content']
        tool_message['content'] = content
        return tool_message, custom_outputs

    def _convert_to_chat_messages(self, messages_generated):
        output_messages = []
        custom_outputs = None
        for message in messages_generated:
            if message['role'] == 'tool':
                message, custom_outputs = self._convert_tool_message(message)
            message['id'] = str(uuid.uuid4())
            output_messages.append(ChatAgentMessage(**message))
        return output_messages, custom_outputs

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}
        last_message = request["messages"][-1]
        chat_history = request["messages"][:-1]

        assistant, user_proxy = self._create_agents(chat_history)
        user_proxy.initiate_chat(assistant, message=last_message['content'], max_turns=5, clear_history=False)

        messages_generated = assistant.chat_messages[user_proxy][len(request["messages"]):]
        output_messages, custom_outputs = self._convert_to_chat_messages(messages_generated)
        return ChatAgentResponse(messages=output_messages, custom_outputs=custom_outputs)

# ✅ Instantiate and register the agent
autogen_model = AutogenAgent(tools)
mlflow.models.set_model(autogen_model)


In [0]:
%pip install pyautogen==0.2.18


In [0]:
dbutils.library.restartPython()

In [0]:
from autogen.oai.client import register_model_client
register_model_client(DatabricksModelServingClient)  # ✅ THIS FIXES THE ERROR


In [0]:
from mlflow.types.agent import ChatAgentMessage

# 🧪 Test: Weather Tool
messages = [ChatAgentMessage(role="user", content="Can you get the weather in San Jose?")]
response = autogen_model.predict(messages=messages)

# 🔍 Print agent messages
print("🔎 Agent Response:\n")
for msg in response.messages:
    print(f"{msg.role}: {msg.content}")

# 🧪 Check for structured tool output
if response.custom_outputs:
    print("\n📦 Custom Outputs:")
    for k, v in response.custom_outputs.items():
        print(f"{k}: {v}")
